In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

import scanpy as sc

from scipy.spatial import distance
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import *


In [ ]:
def compute_PAS_fast(clusterlabel, location, k=10):
    clusterlabel = np.array(clusterlabel)
    location = np.array(location)

    # Fit NearestNeighbors (ignore self-match later)
    nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='auto').fit(location)
    distances, indices = nbrs.kneighbors(location)

    # Remove self (first column is self in most cases)
    neighbor_indices = indices[:, 1:]  # shape: (n_samples, k)

    # Check PAS condition
    mismatches = np.array([
        np.sum(clusterlabel[neighbor_indices[i]] != clusterlabel[i]) > (k / 2)
        for i in range(len(clusterlabel))
    ])

    return np.sum(mismatches) / len(clusterlabel)


def compute_CHAOS_fast(clusterlabel, location):
    clusterlabel = np.array(clusterlabel)
    location = np.array(location)
    matched_location = StandardScaler().fit_transform(location)

    clusterlabel_unique = np.unique(clusterlabel)
    dist_val = 0
    total_count = 0

    for k in tqdm(clusterlabel_unique, desc="Computing CHAOS"):
        cluster_mask = clusterlabel == k
        location_cluster = matched_location[cluster_mask]
        n = location_cluster.shape[0]

        if n <= 2:
            continue

        # Use NearestNeighbors to find 1-NN distances
        nbrs = NearestNeighbors(n_neighbors=2, algorithm='auto').fit(location_cluster)
        distances, _ = nbrs.kneighbors(location_cluster)

        # distances[:, 0] is zero (self), distances[:, 1] is nearest neighbor
        dist_val += np.sum(distances[:, 1])
        total_count += n

    return dist_val / total_count if total_count > 0 else np.nan


def compute_ASW_fast(adata, pred_key, spatial_key='spatial'):
    coords = adata.obsm[spatial_key]
    labels = adata.obs[pred_key]
    return silhouette_score(X=coords, labels=labels, metric='euclidean')

def compute_ARI(adata,gt_key,pred_key):
        return adjusted_rand_score(adata.obs[gt_key],adata.obs[pred_key])

def compute_NMI(adata,gt_key,pred_key):
    return normalized_mutual_info_score(adata.obs[gt_key],adata.obs[pred_key])

def compute_HOM(adata,gt_key,pred_key):
    return homogeneity_score(adata.obs[gt_key],adata.obs[pred_key])

def compute_COM(adata,gt_key,pred_key):
    return completeness_score(adata.obs[gt_key],adata.obs[pred_key])

# Load data

In [ ]:
# --- REQUIRED: set your paths ---
ADATA_PATH = '../../../Broad_SpatialFoundation/notebooks/notebooks_after_review/segmentation/xenium_cellpose.h5ad'
GEOJSON_PATH = "../../../Broad_SpatialFoundation/test_data/10X_Xenium_Ovarian_5k/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_he_annotated_image.geojson"

# --- OPTIONAL: customize these ---
OBSM_KEY   = "spatial_he"                         # obsm key with (x, y) in H&E pixel space
OUT_PATH   = None                                 # None -> will save next to ADATA_PATH with .annotated.h5ad
OUT_COL    = "region_annotation"                  # adata.obs column to write labels into
LABEL_COL  = 'name'                                 # if you know your label field name in the GeoJSON, set it (e.g., "name")
                                                   # otherwise the notebook tries to guess it


In [ ]:
def guess_label_column(gdf: gpd.GeoDataFrame) -> str | None:
    """Pick a reasonable label column from common QuPath exports."""
    candidates = [
        "classification_name",  # from nested 'classification.name'
        "name",
        "label",
        "annotation",
        "class",
        "type",
    ]
    cols = [c for c in gdf.columns if c != gdf.geometry.name]
    for c in candidates:
        if c in cols:
            return c
    return cols[0] if cols else None


def ensure_label_column(gdf: gpd.GeoDataFrame, prefer: str | None) -> str | None:
    """Ensure there's a flat text label column to use."""
    if prefer and prefer in gdf.columns:
        return prefer

    # Some QuPath exports store nested dict in 'classification' with key 'name'
    if "classification" in gdf.columns and "classification_name" not in gdf.columns:
        def _extract_name(v):
            try:
                if isinstance(v, dict):
                    return v.get("name")
                if isinstance(v, str):
                    return json.loads(v).get("name")
            except Exception:
                pass
            return None
        gdf["classification_name"] = gdf["classification"].map(_extract_name)

    return guess_label_column(gdf)


def make_points_gdf(XY: np.ndarray, index, crs) -> gpd.GeoDataFrame:
    """Create a GeoDataFrame of points from (x, y) coordinates."""
    points = gpd.GeoSeries((Point(float(x), float(y)) for x, y in XY),
                           index=index, name="geometry", crs=crs)
    return gpd.GeoDataFrame(geometry=points)


def aggregate_labels(join_df: pd.DataFrame, label_col: str, obs_index) -> pd.Series:
    """Aggregate multiple matches per cell into a semicolon-separated label string."""
    if label_col not in join_df.columns:
        s = join_df.index.to_series().groupby(level=0).size()
        out = pd.Series(index=obs_index, dtype="object")
        out.loc[s.index] = "in_annotation"
        return out

    agg = (
        join_df[label_col]
        .groupby(level=0)
        .apply(lambda s: ";".join(sorted(str(v) for v in set(s.dropna()))))
    )
    out = pd.Series(index=obs_index, dtype="object")
    out.loc[agg.index] = agg
    return out


In [ ]:
def _transform_x(aff_transf: pd.DataFrame, coords: np.ndarray) -> np.ndarray:
    """Why do we need this? The H&E image is not naturally aligned to the Xenium output. This can be done through the
    Xenium
    """

    inv_transf = np.linalg.inv(aff_transf)
    transformed_coords = (inv_transf @ np.vstack((coords.T, np.ones(len(coords))))).T[
        :, :-1
    ]

    return transformed_coords
    
# Alignment matrix from 10X
M = np.array([
    [0.010908748623278200,  1.2895248946320600, -721.007456942807],
    [-1.2895248946320600,  0.010908748623278200, 38642.677876412400],
    [0, 0, 1]
])

In [ ]:
rawdata = sc.read_h5ad(ADATA_PATH)

In [ ]:
rawdata.obs_names = "cell_" + rawdata.obs_names

In [ ]:
coords = rawdata.obsm["spatial_px"]
cell_names = rawdata.obs_names.to_numpy()

transformed_coords = _transform_x(aff_transf=M, coords=coords)
# Clip at 0 bc sometimes the transformation bugs a little bit, this should be minor though
# (ex: 1 of 150,000 cells had this in a dataset I am evaluating)
print(
    f"There are {((transformed_coords<0).sum(axis=1)>0).sum()} cells with negative coordinates, clipping at 0."
)
transformed_coords = transformed_coords.clip(0)


rawdata.obsm['spatial_he'] = transformed_coords

rawdata.obs['X_he'] = rawdata.obsm['spatial_he'][:,0]
rawdata.obs['Y_he'] = rawdata.obsm['spatial_he'][:,1]

In [ ]:
XY = np.asarray(rawdata.obsm[OBSM_KEY])

In [ ]:
ann = gpd.read_file(GEOJSON_PATH)
if ann.empty:
    raise ValueError("Annotation GeoJSON has no geometries.")

ann = ann[ann.geometry.notnull()].copy()
ann = ann[ann.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
if ann.empty:
    raise ValueError("No Polygon/MultiPolygon geometries found in the GeoJSON.")

label_col = ensure_label_column(ann, LABEL_COL)
if label_col is None:
    label_col = "_layer"
    ann[label_col] = "annotation"

print("Annotation label column:", label_col)
display(ann.head())


In [ ]:
# Make sure CRS aligns; QuPath pixel coords commonly have no CRS, which is fine
pts = make_points_gdf(XY, rawdata.obs_names, crs=ann.crs)

# Join with predicate='within' (Shapely 2+ / GeoPandas >=0.10)
joined = gpd.sjoin(pts, ann[[label_col, ann.geometry.name]], how="left", predicate="within")

# Aggregate overlaps and fill missing
labels = aggregate_labels(joined, label_col, obs_index=rawdata.obs_names).fillna("unlabeled")
rawdata.obs[OUT_COL] = pd.Categorical(labels)

print("Done assigning labels. Preview value counts:")
display(rawdata.obs[OUT_COL].value_counts(dropna=False).to_frame("count"))


In [ ]:
rawdata.obs[OUT_COL] = rawdata.obs[OUT_COL].replace({'': 'Unassigned', 'Necrosis;Tumor': 'Necrosis',
                              'Fallopian tube;Smooth muscle': 'Smooth muscle',})

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 10))

regions = rawdata.obs['region_annotation'].unique()
colors = plt.cm.tab10.colors
color_map = {region: colors[i % len(colors)] for i, region in enumerate(regions)}

for region, group in rawdata.obs.groupby('region_annotation'):
    ax.scatter(group['X_he'], group['Y_he'],
               s=0.5, alpha=0.5, color=color_map[region], label=region, rasterized=True)

ax.set_aspect('equal')
ax.legend(markerscale=5, bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Load embeddings

In [ ]:
adata = rawdata.copy()

In [ ]:
emb_uni_scgpt_df = pd.read_parquet('../../../Broad_SpatialFoundation/notebooks/notebooks_after_review/segmentation/uni_scgpt_avg.parquet').set_index('cell_id')

emb_virchow_scgpt_df = pd.read_parquet('../../../Broad_SpatialFoundation/notebooks/notebooks_after_review/segmentation/virchow_scgpt_avg.parquet').set_index('cell_id')

emb_uni_nicheformer_df = pd.read_parquet('../../../Broad_SpatialFoundation/notebooks/notebooks_after_review/segmentation/uni_nicheformer_avg.parquet').set_index('cell_id')

emb_virchow_nicheformer_df = pd.read_parquet('../../../Broad_SpatialFoundation/notebooks/notebooks_after_review/segmentation/virchow_nicheformer_avg.parquet').set_index('cell_id')

In [ ]:
nichecompass_embeddings = pd.read_parquet('../../../Broad_SpatialFoundation/notebooks/notebooks_after_review/segmentation/embeddings/nichecompass.parquet')

nicheformer_embeddings = pd.read_parquet('../../../Broad_SpatialFoundation/notebooks/notebooks_after_review/segmentation/embeddings/nicheformer.parquet')

banksy_embeddings = pd.read_parquet('../../../Broad_SpatialFoundation/notebooks/notebooks_after_review/segmentation/embeddings/banksy_08.parquet')

In [ ]:
nichecompass_embeddings.index = "cell_" + nichecompass_embeddings.index

nicheformer_embeddings.index = "cell_" + nicheformer_embeddings.index

In [ ]:
emb_uni_scgpt_df.columns = emb_uni_scgpt_df.columns.astype(str)

emb_virchow_scgpt_df.columns = emb_virchow_scgpt_df.columns.astype(str)

emb_uni_nicheformer_df.columns = emb_uni_nicheformer_df.columns.astype(str)

emb_virchow_nicheformer_df.columns = emb_virchow_nicheformer_df.columns.astype(str)

In [ ]:
adata.obsm['gcn_uni_scgpt'] = emb_uni_scgpt_df.loc[adata.obs_names,['0','1','2','3','4','5','6','7','8','9']]

adata.obsm['gcn_virchow_scgpt'] = emb_virchow_scgpt_df.loc[adata.obs_names,['0','1','2','3','4','5','6','7','8','9']]

adata.obsm['gcn_uni_nicheformer'] = emb_uni_nicheformer_df.loc[adata.obs_names,['0','1','2','3','4','5','6','7','8','9']]

adata.obsm['gcn_virchow_nicheformer'] = emb_virchow_nicheformer_df.loc[adata.obs_names,['0','1','2','3','4','5','6','7','8','9']]

In [ ]:
adata.obsm['banksy'] = banksy_embeddings.loc[adata.obs_names]

adata.obsm['nichecompass'] = nichecompass_embeddings.loc[adata.obs_names]

adata.obsm['nicheformer'] = nicheformer_embeddings.loc[adata.obs_names]

# Run clustering

In [ ]:
sc.pp.neighbors(adata, use_rep = 'gcn_uni_scgpt')

In [ ]:
sc.tl.leiden(adata, resolution=0.1, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['leiden_gcn_uni_scgpt'] = adata.obs.leiden.replace({'13': '12', '14': '12', '15': '12', '16': '12', '17': '12', '18': '12',
                                                        '19': '12', })

In [ ]:
sc.pp.neighbors(adata, use_rep = 'gcn_virchow_scgpt')

In [ ]:
sc.tl.leiden(adata, resolution=0.1, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['leiden_gcn_virchow_scgpt'] = adata.obs.leiden.replace({'13': '12', '14': '12', '15': '12', '16': '12', '17': '12', '18': '12',
                                                        '19': '12', })

In [ ]:
sc.pp.neighbors(adata, use_rep = 'gcn_uni_nicheformer')

In [ ]:
sc.tl.leiden(adata, resolution=0.1, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['leiden_gcn_uni_nicheformer'] = adata.obs.leiden.replace({'13': '12', '14': '12', '15': '12', '16': '12',})

In [ ]:
sc.pp.neighbors(adata, use_rep = 'gcn_virchow_nicheformer')

In [ ]:
sc.tl.leiden(adata, resolution=0.1, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['leiden_gcn_virchow_nicheformer'] = adata.obs.leiden.replace({'13': '12', '14': '12', '15': '12', })

In [ ]:
sc.pp.neighbors(adata, use_rep = 'nichecompass')

In [ ]:
sc.tl.leiden(adata, resolution=0.35, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['leiden_nichecompass'] = adata.obs.leiden.replace({'13': '12', '14': '12', '15': '12', })

In [ ]:
sc.pp.neighbors(adata, use_rep = 'nicheformer')

In [ ]:
sc.tl.leiden(adata, resolution=0.001, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['leiden_nicheformer'] = adata.obs.leiden.replace({'13': '12', '14': '12', '15': '12', })

In [ ]:
sc.pp.neighbors(adata, use_rep = 'banksy')

In [ ]:
sc.tl.leiden(adata, resolution=0.5, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['leiden_banksy'] = adata.obs.leiden.replace({'13': '12', '14': '12', '15': '12', '16': '12', '17': '12', })

In [ ]:
adata.layers['counts'] = adata.X.copy()

In [ ]:
sc.pp.normalize_total(adata, target_sum=10000)
sc.pp.log1p(adata)

In [ ]:
sc.pp.neighbors(adata)

In [ ]:
sc.tl.leiden(adata, resolution=0.55, flavor="igraph", n_iterations=2)

In [ ]:
# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)

# Make mapping: old → new (ranked by size)
mapping = {old: str(new) for new, old in enumerate(counts.index)}

# Apply mapping
adata.obs['leiden'] = adata.obs['leiden'].map(mapping).astype('category')

# (optional) sort categories by their new numeric label
adata.obs['leiden'].cat.reorder_categories(sorted(adata.obs['leiden'].cat.categories, key=int))

print("Cluster relabeling done ✅")# Get cluster sizes
counts = adata.obs['leiden'].value_counts().sort_values(ascending=False)
display(counts)

In [ ]:
adata.obs['leiden_scanpy'] = adata.obs.leiden.replace({'13': '12',  })

# Now compute

In [ ]:
def compute_all_metrics(adata, clustering_keys, ground_truth_key='path_region', spatial_key='spatial_px'):
    results = {}

    for method_name, cluster_key in clustering_keys.items():
        metrics = {
            'ARI': compute_ARI(adata, cluster_key, ground_truth_key),
            'NMI': compute_NMI(adata, cluster_key, ground_truth_key),
            'HOM': compute_HOM(adata, cluster_key, ground_truth_key),
            'COM': compute_COM(adata, cluster_key, ground_truth_key),
            'PAS': compute_PAS_fast(adata.obs[cluster_key], adata.obsm[spatial_key]),
            'CHAOS': compute_CHAOS_fast(adata.obs[cluster_key], adata.obsm[spatial_key]),
        }
        results[method_name] = metrics

    return pd.DataFrame(results)

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

def format_number(value):
    """Format numbers: scientific notation if <0.01, else 2 decimals."""
    if pd.isna(value):
        return ""
    if abs(value) < 0.01 and value != 0:
        return f"{value:.0e}"  # 1 decimal in scientific notation, e.g. 3.4e-04
    else:
        return f"{value:.2f}"  # two decimals otherwise

def plot_benchmark_heatmap(
    results_df,
    title="Spatial clustering benchmark",
    savefig=None,
    metric_order=None,
):
    """
    Nature Genetics–style benchmarking heatmap showing method rankings across metrics.
    Allows manual control of metric order.
    """

    lower_better = {'PAS', 'CHAOS'}

    # --- Default metric order ---
    if metric_order is None:
        metric_order = list(results_df.index)

    # --- Normalize scores ---
    df_norm = results_df.copy()
    for metric in df_norm.index:
        vals = df_norm.loc[metric]
        if metric in lower_better:
            vals = -vals
        df_norm.loc[metric] = (vals - vals.min()) / (vals.max() - vals.min() + 1e-9)

    # --- Rank per metric ---
    ranks = results_df.copy()
    for metric in ranks.index:
        ranks.loc[metric] = results_df.loc[metric].rank(ascending=(metric in lower_better))

    # --- Prepare longform for plotting ---
    df_plot = df_norm.reset_index().melt(
        id_vars='index', var_name='Method', value_name='Normalized'
    ).rename(columns={'index': 'Metric'})

    df_plot['Raw'] = results_df.reset_index().melt(
        id_vars='index', var_name='Method', value_name='Raw'
    )['Raw']

    df_plot['Rank'] = ranks.reset_index().melt(
        id_vars='index', var_name='Method', value_name='Rank'
    )['Rank']

    # Add directional arrows
    df_plot['MetricLabel'] = df_plot['Metric'].apply(
        lambda m: f"{m} {'↓' if m in lower_better else '↑'}"
    )

    # --- Construct ordered MetricLabel list ---
    metric_order_labels = []
    for m in metric_order:
        arrow = '↓' if m in lower_better else '↑'
        metric_order_labels.append(f"{m} {arrow}")

    # --- Heatmap data matrix ---
    method_order = results_df.columns.tolist()
    df_matrix = df_plot.pivot_table(
        index="MetricLabel", columns="Method", values="Normalized"
    ).loc[metric_order_labels, method_order]

    # --- Aesthetics ---
    sns.set_theme(style="white", context="talk")

    fig, ax = plt.subplots(figsize=(1.3 * len(method_order), 0.8 * len(metric_order)), dpi=300)
    # Enhance contrast near the top (gamma correction)
    gamma = 3  ### THIS IS ONLY FOR THE COLOR FOR PLOTTING PURPOSES, NOT THE NUMBERS!
    df_matrix_contrast = df_matrix ** gamma
    sns.heatmap(
        df_matrix_contrast,
        #cmap="vlag",
        cmap = LinearSegmentedColormap.from_list(
            "vlag_red",
            ["#fee8ef",  # very light pink
             "#f4a3a8",  # pastel red
             "#d95858",  # mid red
             "#b40426"]  # vlag red (vivid crimson)
        ),
        cbar=False,
        ax=ax,
        linewidths=0,
        square=True,
    )

    # --- Adaptive text color (white on dark, black on light) ---
    #cmap = plt.get_cmap("vlag")
    cmap = LinearSegmentedColormap.from_list(
        "vlag_red",
        ["#fee8ef",  # very light pink
         "#f4a3a8",  # pastel red
         "#d95858",  # mid red
         "#b40426"]  # vlag red (vivid crimson)
    )

    for i, metric in enumerate(df_matrix.index):
        base_metric = metric.split()[0]
        for j, method in enumerate(df_matrix.columns):
            raw_val = results_df.loc[base_metric, method]
            norm_val = df_matrix.loc[metric, method]

            # Compute luminance for adaptive color
            rgb = np.array(cmap(norm_val)[:3])
            luminance = 0.2126 * rgb[0] + 0.7152 * rgb[1] + 0.0722 * rgb[2]
            text_color = "black" if luminance > 0.5 else "white"

            ax.text(
                j + 0.5, i + 0.5,
                format_number(raw_val),
                ha='center', va='center',
                color=text_color,
                fontsize=8,
                fontweight='normal',
            )

    # --- Formatting ---
    ax.set_title(title, fontsize=10, pad=14, fontweight='normal')
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=10, fontweight='normal')
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=10, fontweight='normal')

    for spine in ax.spines.values():
        spine.set_visible(False)

    plt.tight_layout()

    if savefig:
        fig.savefig(
            savefig,
            bbox_inches="tight",
            dpi=300,
            format=savefig.split('.')[-1],
            transparent=True
        )
        print(f"Saved: {savefig}")

    plt.show()



In [ ]:
clustering_keys = {
    'SpatialFusion (UNI+scGPT)': 'leiden_gcn_uni_scgpt',
    'SpatialFusion (UNI+Nicheformer)': 'leiden_gcn_uni_nicheformer',
    'SpatialFusion (Virchow+scGPT)': 'leiden_gcn_virchow_scgpt',
    'SpatialFusion (Virchow+Nicheformer)': 'leiden_gcn_virchow_nicheformer',
    'NicheCompass': 'leiden_nichecompass',
    'BANKSY': 'leiden_banksy',
    'Nicheformer': 'leiden_nicheformer',
    'Scanpy': 'leiden_scanpy',

}

results_df = compute_all_metrics(adata, clustering_keys, ground_truth_key='region_annotation', spatial_key='spatial_px')


In [ ]:
import matplotlib
matplotlib.rcParams['svg.fonttype'] = 'none'
plot_benchmark_heatmap(results_df, title="OVCA Benchmark", savefig='CELLPOSE_OVCA_benchmark.svg')